In [ ]:
import numpy as np
from scipy.special import gammaln
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    "text.usetex":      False,
    "font.family":      "serif",
    "font.serif":       ["Palatino", "Palatino Linotype", "Georgia",
                         "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.labelsize":   9,
    "axes.titlesize":   9,
    "xtick.labelsize":  8,
    "ytick.labelsize":  8,
    "figure.dpi":       150,
})

_NAVY   = ( 28/255,  58/255, 110/255)
_SIENNA = (160/255,  90/255,  60/255)
_GREEN  = ( 45/255, 120/255,  80/255)
_VIOLET = (100/255,  50/255, 130/255)
_AMBER  = (185/255, 120/255,  35/255)
_TEAL   = ( 25/255, 110/255, 110/255)

P_COLORS     = [_NAVY, _GREEN, _SIENNA, _VIOLET, _AMBER, _TEAL]
P_LINESTYLES = ["-",   "-",    "-",     "-",      "--",   "--"]

SEED     = 42
Q        = 0.5
P_VALUES = [0.10, 0.30, 0.50, 0.65, 0.70, 0.74]
N_VALUES = [250, 500, 1000, 2500, 5000, 10000, 20000,30000,40000,50000]
M        = 50
B        = 2000
Z95      = 1.96

def simulate_endpoints(p: float, q: float, n: int,
                       total_paths: int,
                       rng: np.random.Generator) -> np.ndarray:
    alpha = 2.0 * p - 1.0

    S = np.where(rng.random(total_paths) < q, 1.0, -1.0)

    for t in range(1, n):
        prob_plus = 0.5 + (alpha / (2.0 * t)) * S
        step = np.where(rng.random(total_paths) < prob_plus, 1.0, -1.0)
        S += step

    return S

def true_mean(n: int, alpha: float, beta: float) -> float:
    if beta == 0.0:
        return 0.0
    log_mu = (np.log(abs(beta))
              + gammaln(n + alpha)
              - gammaln(alpha + 1.0)
              - gammaln(float(n)))
    return float(np.sign(beta) * np.exp(log_mu))

def half_width(n: int, m: int, p: float) -> float:
    return Z95 * np.sqrt(n / (m * (3.0 - 4.0 * p)))

def run_simulation() -> dict:
    beta     = 2.0 * Q - 1.0
    rng      = np.random.default_rng(SEED)
    coverage = {p: [] for p in P_VALUES}

    n_pairs = len(P_VALUES) * len(N_VALUES)
    done    = 0

    for p in P_VALUES:
        alpha = 2.0 * p - 1.0
        for n in N_VALUES:
            mu_n  = true_mean(n, alpha, beta)
            hw    = half_width(n, M, p)

            total = B * M
            S_all = simulate_endpoints(p, Q, n, total, rng)

            S_nm  = S_all.reshape(B, M).mean(axis=1)

            cov = float(np.mean((S_nm - hw <= mu_n) & (mu_n <= S_nm + hw)))
            coverage[p].append(cov)

            done += 1
            print(f"  [{done:2d}/{n_pairs}]  p={p:.2f}  n={n:6d}  "
                  f"coverage={cov:.4f}")

    return coverage

def make_figure(coverage: dict, filename: str = "figure1_empirical_coverage_diffusive"):
    fig, ax = plt.subplots(figsize=(5.4, 3.4))
    fig.subplots_adjust(left=0.13, right=0.97, bottom=0.16, top=0.97)

    x = np.array(N_VALUES, dtype=float)

    for idx, p in enumerate(P_VALUES):
        y = np.array(coverage[p])
        ax.plot(
            x, y,
            color     = P_COLORS[idx],
            linestyle = P_LINESTYLES[idx],
            linewidth = 1.2,
            label     = f"$p = {p}$",
        )

    ax.axhline(0.95, color=(0.3, 0.3, 0.3), linewidth=0.75,
               linestyle=":", label=r"nominal $0.95$")

    ax.set_xscale("log")
    ax.set_xticks(N_VALUES)
    ax.xaxis.set_major_formatter(
        ticker.FuncFormatter(lambda v, _: f"${int(v):,}$".replace(",", r"{,}"))
    )
    ax.xaxis.set_minor_locator(ticker.NullLocator())
    ax.set_xlabel(r"$n$", labelpad=3)

    y_all = [v for vals in coverage.values() for v in vals]
    ax.set_ylim(min(min(y_all) - 0.01, 0.875), 0.97 + 0.008)
    ax.set_ylabel(r"$\widehat{\mathrm{Cov}}_n$", labelpad=5)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))

    ax.legend(
        loc          = "lower right",
        frameon      = True,
        framealpha   = 0.93,
        edgecolor    = (0.78, 0.78, 0.78),
        fontsize     = 8,
        handlelength = 2.0,
        ncol         = 2,
        columnspacing= 1.0,
        borderpad    = 0.6,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")

    fig.savefig(f"{filename}.png", dpi=200, bbox_inches="tight")
    fig.savefig(f"{filename}.pdf",          bbox_inches="tight")
    print(f"\nSaved  {filename}.png  /  {filename}.pdf")
    plt.close(fig)

if __name__ == "__main__":
    import time
    print("ERW Monte Carlo — diffusive coverage study")
    print(f"  q={Q},  m={M},  B={B},  seed={SEED}")
    print(f"  p in {P_VALUES}")
    print(f"  n in {N_VALUES}")
    print(f"  (estimated runtime: 3–6 min depending on CPU)\n")

    t0       = time.time()
    coverage = run_simulation()
    print(f"\nSimulation done in {time.time()-t0:.1f}s.  Generating figure ...")
    make_figure(coverage)
    print("Done.")